# Level E Instructions:

### Some steps to get familiar with the data:

- Have a look at the rgb images in the folder "ARKitScenesData/47333473/47333473_frames/lowres_wide/" to get an impression of the scene that we are working with
- Run the notebook to see the complete pipeline: Ground truth visualization, example detections, and full results

## Level E

The task is to generate 3D bounding boxes that mark the estimated location of objects in the environment.
Using an open-vocabulary object detector (OwlV2), we are not limited to a specific set of objects but can choose the set ourselves without needing to adapt the detector.

Required to pass the level:
- Functional pipeline with visualization of the estimated 3D bounding boxes
- mIoU score > 0.17 of the bounding box estimates when comparing to ground-truth (For classes: "bed", "sofa", "chair", "table", "shelf")
- Understand and explain the code flow and steps required for the complete pipeline
- Show at least one detection of a non-ground-truth object (see last cell)

## 1. Dependencies and Imports


In [ ]:
# Install dependencies
%pip install --upgrade pip
%pip install torch==2.4.0+cu121 torchvision==0.19.0+cu121 --index-url https://download.pytorch.org/whl/cu121
%pip install transformers==4.44.0 huggingface-hub==0.24.0 pillow numpy opencv-python open3d ipympl rerun-sdk[notebook]==0.24.1

In [ ]:
# Import required libraries
import os
from typing import Dict, List
import numpy as np

# Import lab utility functions
from lab_utils.data_utils import get_frame_list, load_camera_poses, validate_and_align_frame_data
from lab_utils.ground_truth import load_ground_truth_data
from lab_utils.tsdf_utils import build_tsdf_point_cloud
from lab_utils.scene_visualization import visualize_3d_scene_bbox_results
from lab_utils.evaluation_utils import evaluate_level_results
from lab_utils.model_loaders import load_owlv2_model
from lab_utils.level_e_viz import visualize_level_e_example
from lab_utils.batch_processing_utils import process_frames_in_batches
from lab_utils.detection_utils import detect_objects_in_frame
from lab_utils.geometry_utils_3d import extract_depth_region, create_3d_bbox_corners

## 2. Configuration

Tune the detector and multi-view merge parameters if useful. OFFICIAL_REQUIRED_MIOU is fixed at mIoU > 0.17 and must not be changed.

The main tunable parameters are `detection_threshold`, `merge_iou_threshold`, and `min_detections_for_merge`.

In [ ]:
OFFICIAL_REQUIRED_MIOU = 0.17

class Config:
    # Student notebook configuration
    CODE_MODE = "student"
    
    # Scene Configuration
    SCENE_ID = "47333473"
    BASE_PATH = f"ARKitScenesData/{SCENE_ID}/{SCENE_ID}_frames"
    RGB_PATH = os.path.join(BASE_PATH, "lowres_wide")
    DEPTH_PATH = os.path.join(BASE_PATH, "lowres_depth")
    INTRINSICS_PATH = os.path.join(BASE_PATH, "lowres_wide_intrinsics")
    TRAJ_FILE_PATH = os.path.join(BASE_PATH, "lowres_wide.traj")
    
    # Detection Classes
    OBJECT_CLASSES = ["bed", "chair", "sofa", "table", "shelf"]
    
    # Level E Configuration
    LEVEL_E_CONFIG = {
        'frame_skip': 3,
        'max_frames': 1000,
        'detection_threshold': 0.6,
        'owl_batch_size': 20,
        'merge_iou_threshold': 0.3,
        'min_detections_for_merge': 4,
        'example_viz_index': 47,
        'required_miou': OFFICIAL_REQUIRED_MIOU
    }
    
# TSDF Configuration
    TSDF_CONFIG = {
        'frame_skip': 3, 'depth_scale': 1000.0, 'depth_trunc': 7.0,
        'voxel_size': 0.04, 'batch_size': 20, 'max_frames': 1000,
        'volume_length': 30.0, 'resolution': 512,
    }

    # GT and Display Configuration
    GT_CONFIG = {
        'allowed_classes': None, 'mesh_downsample_points': 75000,
        'show_mesh': True, 'show_annotations': True
    }
    RERUN_WIDTH, RERUN_HEIGHT = 1200, 500

def validate_config(config: Config) -> None:
    """Validate and initialize configuration parameters."""
    config.GT_CONFIG['allowed_classes'] = config.OBJECT_CLASSES
    
    valid_code_modes = {"student"}
    if config.CODE_MODE not in valid_code_modes:
        raise ValueError(f"CODE_MODE must be one of {valid_code_modes}")
    if not np.isclose(config.LEVEL_E_CONFIG['required_miou'], OFFICIAL_REQUIRED_MIOU):
        raise ValueError("Official Level E mIoU threshold is fixed at 0.17")
    
    print(f"Configuration validated - Code mode: {config.CODE_MODE}")
    print(f"Official grading threshold: mIoU > {OFFICIAL_REQUIRED_MIOU:.2f} (fixed)")
    print("Tunable: detection_threshold, merge_iou_threshold, min_detections_for_merge")

# Create and validate config
config = Config()
validate_config(config)

## 3. Ground Truth Visualization

Let's start by visualizing the ground truth data to understand what we're working with:

In [ ]:
# Load and visualize ground truth to understand the scene
print("Loading ground truth data to understand our scene...")

gt_annotations, gt_mesh = load_ground_truth_data(
    config.SCENE_ID, 
    config.BASE_PATH,
    config.GT_CONFIG
)

if gt_annotations:
    print(f"✓ Loaded {len(gt_annotations)} ground truth annotations")
    class_counts = {}
    for ann in gt_annotations:
        class_counts[ann['label']] = class_counts.get(ann['label'], 0) + 1
    print(f"Objects in scene: {class_counts}")

if gt_mesh:
    print(f"✓ Loaded ground truth mesh with {len(gt_mesh.points)} points")

# Visualize the ground truth scene
visualize_3d_scene_bbox_results(
    point_cloud=None,
    detections_3d=None,
    gt_annotations=gt_annotations,
    gt_mesh=gt_mesh,
    show_ground_truth=True,
    show_gt_mesh=True,
    show_object_pointclouds=False,
    title=f"Ground Truth Scene {config.SCENE_ID} - What We Want to Detect",
    config=config
)

print("This shows the ground truth objects we want to detect!")

## 4. Execution Functions

These functions orchestrate the different parts of the pipeline:

In [ ]:
def run_ground_truth_visualization(config: Config) -> None:
    """Execute ground truth visualization."""
    print("=" * 60)
    print("GROUND TRUTH VISUALIZATION")
    print("=" * 60)
    
    gt_annotations, gt_mesh = load_ground_truth_data(
        config.SCENE_ID, 
        config.BASE_PATH,
        config.GT_CONFIG
    )
    
    if gt_annotations:
        print(f"Loaded {len(gt_annotations)} ground truth annotations")
        class_counts = {}
        for ann in gt_annotations:
            class_counts[ann['label']] = class_counts.get(ann['label'], 0) + 1
        print(f"GT objects by class: {class_counts}")
    
    if gt_mesh:
        print(f"Loaded ground truth mesh with {len(gt_mesh.points)} points")
    
    visualize_3d_scene_bbox_results(
        point_cloud=None,
        detections_3d=None,
        gt_annotations=gt_annotations,
        gt_mesh=gt_mesh,
        show_ground_truth=True,
        show_gt_mesh=True,
        show_object_pointclouds=False,
        title=f"Ground Truth Only - Scene {config.SCENE_ID}",
        config=config
    )
    
    print("Ground truth visualization complete!")


def run_example_visualization(config: Config) -> Dict:
    """Execute example visualization."""
    print("\n" + "=" * 60)
    print("EXAMPLE VISUALIZATION")
    print("=" * 60)
    
    example_results = visualize_level_e_example(
        config, 
        frame_index=config.LEVEL_E_CONFIG['example_viz_index'],
        show_depth_analysis=True
    )
    
    print("Example visualization complete!")
    return example_results


def run_full_pipeline(config: Config) -> Dict:
    """Execute the complete 3D scene analysis pipeline."""
    validate_config(config)
    print("\n" + "=" * 60)
    print("FULL PIPELINE EXECUTION")
    print("=" * 60)
    
    processor, model, device = load_owlv2_model()
    camera_poses = load_camera_poses(config.TRAJ_FILE_PATH)
    frames_metadata = get_frame_list(config.RGB_PATH, config.LEVEL_E_CONFIG['frame_skip'])
    aligned_frames = validate_and_align_frame_data(
        frames_metadata, camera_poses, config.RGB_PATH, 
        config.DEPTH_PATH, config.INTRINSICS_PATH, timestamp_tolerance=0.1
    )
    
    if not aligned_frames:
        print("ERROR: No aligned frames found! Check data paths.")
        return {'detections_3d': [], 'frame_results': {}, 'statistics': {}}
    
    frames_for_detection = aligned_frames[:config.LEVEL_E_CONFIG['max_frames']]
    raw_detections_3d, frame_results, detection_stats = process_frames_in_batches(
        frames_for_detection, config, processor, model, device
    )
    
    merged_detections = merge_overlapping_detections(
        raw_detections_3d,
        iou_threshold=config.LEVEL_E_CONFIG['merge_iou_threshold'],
        min_detections_for_merge=config.LEVEL_E_CONFIG['min_detections_for_merge']
    ) if raw_detections_3d else []
    
    print(f"Pipeline: {len(frames_for_detection)} frames → {len(raw_detections_3d)} raw → {len(merged_detections)} merged")
    
    tsdf_point_cloud = build_tsdf_point_cloud(config, max_frames_for_mapping=596, use_cached=True)
    gt_annotations, gt_mesh = load_ground_truth_data(config.SCENE_ID, config.BASE_PATH, config.GT_CONFIG)
    eval_results = evaluate_level_results(merged_detections, gt_annotations, "Level E (OWLv2 Only)", required_miou=config.LEVEL_E_CONFIG['required_miou'], box_iou_fn=compute_3d_bbox_iou)
    
    print(f"Results: {'✓ PASSED' if eval_results['passed'] else '✗ FAILED'} | "
          f"mIoU: {eval_results['mean_iou']:.3f} | Detections: {eval_results['num_detections']}")
    
    if merged_detections or raw_detections_3d or tsdf_point_cloud:
        visualize_3d_scene_bbox_results(
            point_cloud=tsdf_point_cloud, detections_3d=merged_detections,
            raw_detections_3d=raw_detections_3d, gt_annotations=gt_annotations,
            gt_mesh=None, show_ground_truth=True, show_gt_mesh=False,
            show_object_pointclouds=False, show_raw_detections=True,
            title=f"Level E: Raw + Merged Detections - Scene {config.SCENE_ID}",
            config=config
        )
    
    print("Full pipeline complete!")
    return {
        'detections_3d': merged_detections,
        'raw_detections_3d': raw_detections_3d,
        'frame_results': frame_results,
        'statistics': {
            **detection_stats,
            'total_3d_detections_merged': len(merged_detections),
            'detection_classes': list(detection_stats['detection_classes']),
            'alignment_success_rate': len(aligned_frames) / len(frames_metadata) * 100 if frames_metadata else 0,
            'merge_ratio': len(merged_detections) / len(raw_detections_3d) if raw_detections_3d else 0
        },
        'evaluation': eval_results
    }

## 5. 2D Detection with OWLv2 Model

OWLv2 open-vocabulary detection (`detect_objects_in_frame`) is provided — run the visualization below to see it in action before we turn 2D boxes into 3D ones.
Hint: you might have to tune the detection parameters in the config (LEVEL_E_CONFIG).

Visualize the detector:

In [ ]:
# Run example visualization to see detection process on a single frame
print("Running example detection on a single frame...")

example_results = visualize_level_e_example(
    config, 
    frame_index=config.LEVEL_E_CONFIG['example_viz_index'],
    show_depth_analysis=True
)

## 6. Camera Projection (Sensor Fusion)

Implement TODO 1 below: turning a single 2D detection + a depth value into a 3D point via the pinhole camera model. `generate_3d_detections` (provided) uses your implementation to build full 3D boxes for every detection.

In [ ]:
# TODO 1: Camera Projection
def project_pixel_to_3d(center_pixel: List[float], depth: float, camera_intrinsics: np.ndarray) -> List[float]:
    """Project a single pixel + depth to 3D camera coordinates."""
    # Use the camera intrinsics and the pinhole-camera convention. The result must be [x, y, z] in the camera
    # frame, with z equal to the supplied depth.

    # Placeholders (replace with your implementation above)
    x_3d, y_3d, z_3d = 0.0, 0.0, depth

    return [float(x_3d), float(y_3d), float(z_3d)]



def generate_3d_detections(detections_2d: List[Dict],
                          depth_image: np.ndarray,
                          camera_intrinsics: np.ndarray,
                          camera_pose: np.ndarray) -> List[Dict]:
    """Generate 3D detections from 2D detections."""
    detections_3d = []
    
    for detection in detections_2d:
        valid_depths, depth_stats = extract_depth_region(
            detection['bbox'], 
            depth_image
        )
        
        if valid_depths is None or depth_stats['valid_pixels'] < 10:
            continue
        if depth_stats['mean'] < 0.2 or depth_stats['mean'] > 10.0:
            continue
        
        x1, y1, x2, y2 = detection['bbox']
        center_pixel = [(x1 + x2) / 2.0, (y1 + y2) / 2.0]
        
        depth_center = np.median(valid_depths)
        
        # Uses your project_pixel_to_3d implementation:
        center_3d_camera = project_pixel_to_3d(center_pixel, depth_center, camera_intrinsics)
        
        fx, fy = camera_intrinsics[0, 0], camera_intrinsics[1, 1]
        width_3d = (x2 - x1) * depth_center / fx
        height_3d = (y2 - y1) * depth_center / fy
        depth_range = depth_stats['max'] - depth_stats['min']
        depth_3d = max(0.2, depth_range)
        
        bbox_3d_camera = create_3d_bbox_corners(center_3d_camera, width_3d, height_3d, depth_3d)
        
        camera_pose_inv = np.linalg.inv(camera_pose)
        
        center_3d_cam_hom = np.array(center_3d_camera + [1])
        center_3d_world = (camera_pose_inv @ center_3d_cam_hom)[:3].tolist()
        
        bbox_3d_cam = np.array(bbox_3d_camera)
        bbox_3d_cam_hom = np.concatenate([bbox_3d_cam, np.ones((8, 1))], axis=1)
        bbox_3d_world_hom = (camera_pose_inv @ bbox_3d_cam_hom.T).T
        bbox_3d_world = bbox_3d_world_hom[:, :3].tolist()
        
        detection_3d = {
            'label': detection['label'],
            'score': detection['score'],
            'bbox_2d': detection['bbox'],
            'center_3d_world': center_3d_world,
            'bbox_3d_world': bbox_3d_world,
            'depth_stats': depth_stats
        }
        
        detections_3d.append(detection_3d)
    
    return detections_3d

## 7. Multi-View Fusion via 3D IoU

Implement TODO 2 below: volumetric IoU between two 3D boxes. `find_overlapping_detections` and `merge_overlapping_detections` (provided) use it to fuse repeated detections of the same object seen from different frames.

In [ ]:
# TODO 2: 3D Bounding Box IoU
def compute_3d_bbox_iou(bbox1: List[List[float]], bbox2: List[List[float]]) -> float:
    """Compute volumetric IoU between two 3D boxes, each given as a list of 8 corners."""
    # ============================================================================
    # TODO: Implement volumetric IoU for the axis-aligned bounds represented by
    # the two corner sets. The result must be in [0, 1], be zero for disjoint
    # or degenerate boxes, and remain safe when the union has zero volume.
    # Keep the calculation independent of the ordering of the eight corners.
    # ============================================================================
    try:
        corners1 = np.array(bbox1)
        corners2 = np.array(bbox2)

        min1, max1 = None, None  # TODO: per-axis bounds of bbox1
        min2, max2 = None, None  # TODO: per-axis bounds of bbox2

        intersection_min = None  # TODO
        intersection_max = None  # TODO

        # TODO: if there's no overlap on any axis, return 0.0 here

        intersection_volume = None  # TODO
        volume1 = None  # TODO
        volume2 = None  # TODO
        union_volume = None  # TODO

        # Placeholder - replace with your implementation
        return 0.0

    except Exception:
        return 0.0



In [ ]:
# Quick self-check for compute_3d_bbox_iou (run after implementing it above)
_test_box_a = [[0,0,0],[1,0,0],[0,1,0],[1,1,0],[0,0,1],[1,0,1],[0,1,1],[1,1,1]]
_test_box_b = [[5,5,5],[6,5,5],[5,6,5],[6,6,5],[5,5,6],[6,5,6],[5,6,6],[6,6,6]]
_test_box_c = [[0.5,0.5,0.5],[1.5,0.5,0.5],[0.5,1.5,0.5],[1.5,1.5,0.5],
               [0.5,0.5,1.5],[1.5,0.5,1.5],[0.5,1.5,1.5],[1.5,1.5,1.5]]

assert abs(compute_3d_bbox_iou(_test_box_a, _test_box_a) - 1.0) < 1e-6, "IoU of a box with itself should be 1.0"
assert compute_3d_bbox_iou(_test_box_a, _test_box_b) == 0.0, "Disjoint boxes should have IoU 0.0"
assert 0.0 < compute_3d_bbox_iou(_test_box_a, _test_box_c) < 1.0, "Partially overlapping boxes should have 0 < IoU < 1"
print("Compute_3d_bbox_iou sanity checks passed")

In [ ]:
def find_overlapping_detections(detections_3d: List[Dict], class_name: str, iou_threshold: float = 0.4) -> List[List[int]]:
    """Find which detections of same class overlap significantly."""
    class_detections = [i for i, d in enumerate(detections_3d) if d['label'] == class_name]
    
    overlapping_pairs = []
    
    for i in range(len(class_detections)):
        for j in range(i + 1, len(class_detections)):
            idx1, idx2 = class_detections[i], class_detections[j]
            
            iou = compute_3d_bbox_iou(
                detections_3d[idx1]['bbox_3d_world'],
                detections_3d[idx2]['bbox_3d_world']
            )
            
            if iou >= iou_threshold:
                overlapping_pairs.append([idx1, idx2])
    
    return overlapping_pairs


def merge_detection_cluster(detection_indices: List[int], all_detections: List[Dict]) -> Dict:
    """Merge a cluster of overlapping detections into one."""
    cluster_detections = [all_detections[i] for i in detection_indices]
    
    all_corners = [det['bbox_3d_world'] for det in cluster_detections]
    all_scores = [det['score'] for det in cluster_detections]
    
    merged_corners = np.mean(all_corners, axis=0)
    avg_score = np.mean(all_scores)
    
    class_name = cluster_detections[0]['label']
    
    corners_array = np.array(merged_corners)
    center = np.mean(corners_array, axis=0)
    
    return {
        'label': class_name,
        'score': float(avg_score),
        'bbox_3d_world': merged_corners.tolist(),
        'center_3d_world': center.tolist(),
        'merge_info': {
            'num_detections_merged': len(detection_indices),
            'original_confidences': all_scores
        }
    }


def merge_overlapping_detections(detections_3d: List[Dict], 
                                iou_threshold: float = 0.4,
                                min_detections_for_merge: int = 2) -> List[Dict]:
    """Merge overlapping 3D detections."""
    if not detections_3d:
        return []
    
    print("Merging overlapping detections...")
    
    detections_by_class = {}
    for det in detections_3d:
        class_name = det['label']
        if class_name not in detections_by_class:
            detections_by_class[class_name] = []
        detections_by_class[class_name].append(det)
    
    merged_detections = []
    
    for class_name, class_detections in detections_by_class.items():
        if len(class_detections) < min_detections_for_merge:
            continue
        
        overlapping_pairs = find_overlapping_detections(detections_3d, class_name, iou_threshold)
        
        if not overlapping_pairs:
            continue
        
        processed = set()
        for pair in overlapping_pairs:
            idx1, idx2 = pair
            if idx1 not in processed and idx2 not in processed:
                cluster = set([idx1, idx2])
                
                expanded = True
                while expanded:
                    expanded = False
                    for other_pair in overlapping_pairs:
                        other_idx1, other_idx2 = other_pair
                        if other_idx1 in cluster and other_idx2 not in cluster:
                            cluster.add(other_idx2)
                            expanded = True
                        elif other_idx2 in cluster and other_idx1 not in cluster:
                            cluster.add(other_idx1)
                            expanded = True
                
                if len(cluster) >= min_detections_for_merge:
                    merged_detection = merge_detection_cluster(list(cluster), detections_3d)
                    merged_detections.append(merged_detection)
                    processed.update(cluster)
    
    print(f"Merged {len(detections_3d)} detections into {len(merged_detections)} objects")
    return merged_detections

## 8. Full Pipeline Execution

Now let's run the complete pipeline across all frames. You can enable and disable visualization of different things (GT, raw detections, merged, ...) in the viewer.

In [ ]:
# Execute the complete 3D object detection pipeline
print("Running full pipeline across all frames...")

pipeline_results = run_full_pipeline(config)

## 9. Open Vocabulary Exploration

In [ ]:
# First complete the main pipeline above, then set RUN_EXPLORATION = True to experiment

RUN_EXPLORATION = False  # TODO: Set to True when ready to explore

EXPLORATION_CLASSES = [
    "bed", "chair", "sofa", "table", "shelf",  # Keep original classes
    # TODO: Add 2-3 additional objects you want to find:
    # "obj1",
    # "obj2"
]

if RUN_EXPLORATION:
    if len(EXPLORATION_CLASSES) > 5:
        print(f"Exploring: {EXPLORATION_CLASSES}")
        
        # Create new config with exploration classes
        exploration_config = Config()
        exploration_config.OBJECT_CLASSES = EXPLORATION_CLASSES
        
        # Run pipeline
        results = run_full_pipeline(exploration_config)
        
        # Show what was found
        detections = results.get('detections_3d', [])
        print(f"Found {len(detections)} objects: {[d['label'] for d in detections]}")
    else:
        print("Add some new objects to EXPLORATION_CLASSES to start exploring!")
else:
    print("Set RUN_EXPLORATION = True to try detecting different objects")